In [6]:
#任务一

import os
import pandas as pd
from pathlib import Path


def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        score[i][0] = i * gap
    for j in range(n + 1):
        score[0][j] = j * gap

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            diag = score[i-1][j-1] + s
            up = score[i-1][j] + gap
            left = score[i][j-1] + gap
            score[i][j] = max(diag, up, left)

    # 回溯
    a1, a2 = [], []
    i, j = m, n
    while i > 0 or j > 0:
        current = score[i][j]
        if i > 0 and j > 0:
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            if current == score[i-1][j-1] + s:
                a1.append(seq1[i-1])
                a2.append(seq2[j-1])
                i -= 1
                j -= 1
                continue
        if i > 0 and current == score[i-1][j] + gap:
            a1.append(seq1[i-1])
            a2.append('-')
            i -= 1
            continue
        if j > 0 and current == score[i][j-1] + gap:
            a1.append('-')
            a2.append(seq2[j-1])
            j -= 1

    return ''.join(reversed(a1)), ''.join(reversed(a2)), score[m][n]

INPUT_CSV = "week2_sequence_alignment/data/demo_pairs.csv"
YOUR_NAME = "dujiayi"                  
OUTPUT_DIR = f"submissions/{YOUR_NAME}/week2/results"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "week2_global_alignment_results.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(INPUT_CSV)


results = []
for _, row in df.iterrows():
    seq1, seq2 = row["seq1"], row["seq2"]
    ali1, ali2, final_score = needleman_wunsch(seq1, seq2)
    results.append({
        "pair_id": row["pair_id"],
        "algorithm": "Needleman-Wunsch",
        "match": 1,
        "mismatch": -1,
        "gap": -2,
        "score": final_score,
        "aligned_seq1": ali1,
        "aligned_seq2": ali2
    })

result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_CSV, index=False)
#print(f" 全局比对结果已保存到: {OUTPUT_CSV}")

 全局比对结果已保存到: submissions/dujiayi/week2/results\week2_global_alignment_results.csv


In [7]:
#任务二

import os
import pandas as pd
from pathlib import Path

def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    max_score = 0
    max_i, max_j = 0, 0

    # 局部比对矩阵填充（第一行第一列保持 0）
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            diag = score[i-1][j-1] + s
            up = score[i-1][j] + gap
            left = score[i][j-1] + gap
            score[i][j] = max(diag, up, left, 0)

            if score[i][j] > max_score:
                max_score = score[i][j]
                max_i, max_j = i, j

    # 回溯
    a1, a2 = [], []
    i, j = max_i, max_j
    while i > 0 and j > 0 and score[i][j] != 0:
        current = score[i][j]
        s = match if seq1[i-1] == seq2[j-1] else mismatch

        if current == score[i-1][j-1] + s:
            a1.append(seq1[i-1])
            a2.append(seq2[j-1])
            i -= 1
            j -= 1
        elif current == score[i-1][j] + gap:
            a1.append(seq1[i-1])
            a2.append('-')
            i -= 1
        elif current == score[i][j-1] + gap:
            a1.append('-')
            a2.append(seq2[j-1])
            j -= 1
        else:
            break

    return ''.join(reversed(a1)), ''.join(reversed(a2)), max_score
INPUT_CSV = "week2_sequence_alignment/data/demo_pairs.csv"
YOUR_NAME = "dujiayi"                  
OUTPUT_DIR = f"submissions/{YOUR_NAME}/week2/results"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "week2_local_alignment_results.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(INPUT_CSV)


results = []
for _, row in df.iterrows():
    seq1, seq2 = row["seq1"], row["seq2"]
    ali1, ali2, final_score = needleman_wunsch(seq1, seq2)
    results.append({
        "pair_id": row["pair_id"],
        "algorithm": "Needleman-Wunsch",
        "match": 1,
        "mismatch": -1,
        "gap": -2,
        "score": final_score,
        "aligned_seq1": ali1,
        "aligned_seq2": ali2
    })

result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_CSV, index=False)
#print(f" 全局比对结果已保存到: {OUTPUT_CSV}")


 全局比对结果已保存到: submissions/dujiayi/week2/results\week2_local_alignment_results.csv


In [16]:
#任务四

import csv
import os

# 1. 定义参数组
param_sets = {
    'A': {'match': 1, 'mismatch': -1, 'gap': -1},
    'B': {'match': 1, 'mismatch': -1, 'gap': -2},  # 默认
    'C': {'match': 2, 'mismatch': -1, 'gap': -2},
    'D': {'match': 2, 'mismatch': -2, 'gap': -3}
}

# Needleman-Wunsch 全局比对算法
def needleman_wunsch(seq1, seq2, match_score, mismatch_score, gap_score):
    n, m = len(seq1), len(seq2)
    # 初始化得分矩阵
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1):
        dp[i][0] = gap_score * i
    for j in range(m+1):
        dp[0][j] = gap_score * j
    # 填充矩阵
    for i in range(1, n+1):
        for j in range(1, m+1):
            match = dp[i-1][j-1] + (match_score if seq1[i-1]==seq2[j-1] else mismatch_score)
            delete = dp[i-1][j] + gap_score
            insert = dp[i][j-1] + gap_score
            dp[i][j] = max(match, delete, insert)
    # 回溯得到对齐
    align1, align2 = '', ''
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + (match_score if seq1[i-1]==seq2[j-1] else mismatch_score):
            align1 = seq1[i-1] + align1
            align2 = seq2[j-1] + align2
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + gap_score:
            align1 = seq1[i-1] + align1
            align2 = '-' + align2
            i -= 1
        else:
            align1 = '-' + align1
            align2 = seq2[j-1] + align2
            j -= 1
    return dp[n][m], align1, align2

# Smith-Waterman 局部比对算法
def smith_waterman(seq1, seq2, match_score, mismatch_score, gap_score):
    n, m = len(seq1), len(seq2)
    max_score = 0
    max_pos = (0, 0)
    # 初始化得分矩阵
    dp = [[0]*(m+1) for _ in range(n+1)]
    # 填充矩阵
    for i in range(1, n+1):
        for j in range(1, m+1):
            match = dp[i-1][j-1] + (match_score if seq1[i-1]==seq2[j-1] else mismatch_score)
            delete = dp[i-1][j] + gap_score
            insert = dp[i][j-1] + gap_score
            dp[i][j] = max(0, match, delete, insert)
            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_pos = (i, j)
    align1, align2 = '', ''
    i, j = max_pos
    while dp[i][j] > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + (match_score if seq1[i-1]==seq2[j-1] else mismatch_score):
            align1 = seq1[i-1] + align1
            align2 = seq2[j-1] + align2
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + gap_score:
            align1 = seq1[i-1] + align1
            align2 = '-' + align2
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + gap_score:
            align1 = '-' + align1
            align2 = seq2[j-1] + align2
            j -= 1
        else:
            break
    return max_score, align1, align2

def read_demo_pairs(file_path):
    pairs = []
    with open(file_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row['pair_id'], row['seq1'], row['seq2']))
    return pairs

def calc_alignment_stats(align1, align2):
    alignment_length = len(align1)
    matches = sum(1 for a, b in zip(align1, align2) if a == b and a != '-')
    # 修复：把 a == '-' 改成 b == '-'
    gaps = sum(1 for a in align1 if a == '-') + sum(1 for b in align2 if b == '-')
    return alignment_length, matches, gaps

if __name__ == '__main__':
    
    input_path = 'week2_sequence_alignment/data/demo_pairs.csv'
    YOUR_NAME = "dujiayi"
    output_dir = f"submissions/{YOUR_NAME}/week2/results"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'week2_parameter_sensitivity.csv')

    pairs = read_demo_pairs(input_path)
    algorithms = [('NW', needleman_wunsch), ('SW', smith_waterman)]

    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'pair_id', 'algorithm', 'param_set', 'match', 'mismatch', 'gap',
            'score', 'aligned_seq1', 'aligned_seq2',
            'alignment_length', 'number_of_matches', 'number_of_gaps'
        ])

        for pair_id, seq1, seq2 in pairs:
            for alg_name, alg_func in algorithms:
                for param_set, params in param_sets.items():
                    score, align1, align2 = alg_func(seq1, seq2, params['match'], params['mismatch'], params['gap'])
                    align_len, matches, gaps = calc_alignment_stats(align1, align2)
                    writer.writerow([
                        pair_id, alg_name, param_set,
                        params['match'], params['mismatch'], params['gap'],
                        score, align1, align2,
                        align_len, matches, gaps
                    ])
   # print(f"结果已保存至: {output_path}")


结果已保存至: submissions/dujiayi/week2/results\week2_parameter_sensitivity.csv


In [ ]:
#任务五

import os
import numpy as np
import matplotlib.pyplot as plt

# 序列 pair_001
seq1 = "GATTACA"
seq2 = "GCATGCU"
match = 1
mismatch = -1
gap = -2

# 全局比对 NW 矩阵 
def compute_nw_matrix(seq1, seq2, match, mismatch, gap):
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1):
        dp[i][0] = gap * i
    for j in range(n+1):
        dp[0][j] = gap * j
    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            dp[i][j] = max(
                dp[i-1][j-1] + s,
                dp[i-1][j] + gap,
                dp[i][j-1] + gap
            )
    return dp

def compute_sw_matrix(seq1, seq2, match, mismatch, gap):
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m+1, n+1), dtype=int)
    max_score = 0
    max_pos = (0, 0)
    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            dp[i][j] = max(
                0,
                dp[i-1][j-1] + s,
                dp[i-1][j] + gap,
                dp[i][j-1] + gap
            )
            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_pos = (i, j)
    return dp, max_pos

def plot_matrix(dp, seq1, seq2, title, save_path):
    plt.figure(figsize=(8, 6))
    plt.imshow(dp, cmap='YlGnBu', aspect='equal')

    # 标注每个格子的数值
    for i in range(dp.shape[0]):
        for j in range(dp.shape[1]):
            plt.text(j, i, dp[i, j], ha='center', va='center', fontsize=10, color='black')

    plt.xticks(range(len(seq2)+1), ['-'] + list(seq2))
    plt.yticks(range(len(seq1)+1), ['-'] + list(seq1))
    plt.title(title, fontsize=12)
    plt.colorbar()
    plt.tight_layout()

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300)
    plt.close()

if __name__ == "__main__":
    YOUR_NAME = "dujiayi"
    OUT_FIG = f"submissions/{YOUR_NAME}/week2/figures"

    # 全局矩阵
    nw = compute_nw_matrix(seq1, seq2, match, mismatch, gap)
    plot_matrix(nw, seq1, seq2,
                "Needleman-Wunsch Global DP Matrix",
                os.path.join(OUT_FIG, "week2_global_dp_matrix.png"))

    # 局部矩阵
    sw, _ = compute_sw_matrix(seq1, seq2, match, mismatch, gap)
    plot_matrix(sw, seq1, seq2,
                "Smith-Waterman Local DP Matrix",
                os.path.join(OUT_FIG, "week2_local_dp_matrix.png"))

   # print("任务 5 画图完成！已保存到：", OUT_FIG)

In [6]:
from Bio import Align

#NW 函数
def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1):
        dp[i][0] = gap * i
    for j in range(n+1):
        dp[0][j] = gap * j
        
    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if seq1[i-1]==seq2[j-1] else mismatch
            dp[i][j] = max(dp[i-1][j-1]+s, dp[i-1][j]+gap, dp[i][j-1]+gap)
    
    a1, a2 = [], []
    i, j = m, n
    while i>0 or j>0:
        if i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+(match if seq1[i-1]==seq2[j-1] else mismatch):
            a1.append(seq1[i-1])
            a2.append(seq2[j-1])
            i-=1
            j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+gap:
            a1.append(seq1[i-1])
            a2.append('-')
            i-=1
        else:
            a1.append('-')
            a2.append(seq2[j-1])
            j-=1
    return "", ''.join(reversed(a1)), ''.join(reversed(a2)), dp[m][n], ""

#SW 函数
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    score = [[0]*(n+1) for _ in range(m+1)]
    max_score = 0
    max_i, max_j = 0, 0

    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if seq1[i-1]==seq2[j-1] else mismatch
            diag = score[i-1][j-1] + s
            up = score[i-1][j] + gap
            left = score[i][j-1] + gap
            score[i][j] = max(diag, up, left, 0)
            if score[i][j] > max_score:
                max_score = score[i][j]
                max_i, max_j = i, j

    a1, a2 = [], []
    i, j = max_i, max_j
    while i>0 and j>0 and score[i][j] !=0:
        current = score[i][j]
        s = match if seq1[i-1]==seq2[j-1] else mismatch
        if current == score[i-1][j-1]+s:
            a1.append(seq1[i-1])
            a2.append(seq2[j-1])
            i-=1
            j-=1
        elif current == score[i-1][j]+gap:
            a1.append(seq1[i-1])
            a2.append('-')
            i-=1
        else:
            a1.append('-')
            a2.append(seq2[j-1])
            j-=1
    return "", ''.join(reversed(a1)), ''.join(reversed(a2)), max_score, "", ""

# Biopython 
def biopython_score_example(seq1: str, seq2: str, mode="global"):
    aligner = Align.PairwiseAligner()
    aligner.mode = mode
    aligner.match_score = 1
    aligner.mismatch_score = -1
    
    try:
        aligner.gap_score = -2
    except Exception:
        aligner.open_gap_score = -2
        aligner.extend_gap_score = -2
        
    alignments = aligner.align(seq1, seq2)
    return alignments.score

# 选择两组序列验证 
test_pairs = [
    {"pair_id": "pair_001", "seq1": "GATTACA", "seq2": "GCATGCU"},
    {"pair_id": "pair_003", "seq1": "TTACGTAA", "seq2": "GGACGTCC"}
]

# 输出表格 
print("="*60)
print(f"{'序列对':^10} | {'比对模式':^6} | {'手写得分':^6} | {'Biopython得分':^10} | {'验证状态':^6}")
print("="*60)

for p in test_pairs:
    s1, s2 = p["seq1"], p["seq2"]
    
    # NW
    _, al1, al2, my_nw_score, _ = needleman_wunsch(s1, s2, match=1, mismatch=-1, gap=-2)
    bio_nw_score = biopython_score_example(s1, s2, mode="global")
    status_nw = "PASS " if my_nw_score == bio_nw_score else "FAIL "
    print(f"{p['pair_id']:^12} | {'NW (Global)':^9} | {my_nw_score:^8} | {bio_nw_score:^12.1f} | {status_nw:^6}")
    
    # SW
    _, al1, al2, my_sw_score, _, _ = smith_waterman(s1, s2, match=1, mismatch=-1, gap=-2)
    bio_sw_score = biopython_score_example(s1, s2, mode="local")
    status_sw = "PASS " if my_sw_score == bio_sw_score else "FAIL "
    print(f"{p['pair_id']:^12} | {'SW (Local)':^9} | {my_sw_score:^8} | {bio_sw_score:^12.1f} | {status_sw:^6}")
    
print("="*60)

   序列对     |  比对模式  |  手写得分  | Biopython得分 |  验证状态 
  pair_001   | NW (Global) |    -1    |     -1.0     | PASS  
  pair_001   | SW (Local) |    2     |     2.0      | PASS  
  pair_003   | NW (Global) |    0     |     0.0      | PASS  
  pair_003   | SW (Local) |    4     |     4.0      | PASS  
